# Load & Inspect the data

# import pandas as pd

df = pd.read_csv("messy_data.csv")

print(df.head())
print("\nShape:", df.shape)
print("\nData Types:")
print(df.dtypes)

In [3]:
# Data missing per column
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100
})

print(missing_summary)

                   Missing Count  Missing Percentage
Employee_ID                    0            0.000000
Name                           0            0.000000
Age                            0            0.000000
Department                     0            0.000000
Salary                         1            3.846154
Joining_Date                   0            0.000000
Experience_Years               1            3.846154
Performance_Score              1            3.846154


# Column cleaning 

## Adding median to missing values

In [12]:
# Missing numerical values were imputed using the median because the dataset contains potential outliers, making the median more robust than the mean.


# Salary is of object type, we can't apply median() 
# If a value can't be converted to number, than make it NaN.
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")

salary_median = df["Salary"].median()
print(salary_median)
exp_median = df["Experience_Years"].median()
print(exp_median)
score_median = df["Performance_Score"].median()
print(score_median)


53500.0
4.0
8.2


In [15]:
# Filling the missing values with median

df["Salary"] = df["Salary"].fillna(salary_median)

df["Experience_Years"] = df["Experience_Years"].fillna(exp_median)

df["Performance_Score"] = df["Performance_Score"].fillna(score_median)

In [16]:
# Checking the missing values
print(df.isnull().sum())

Employee_ID          0
Name                 0
Age                  0
Department           0
Salary               0
Joining_Date         0
Experience_Years     0
Performance_Score    0
dtype: int64


## Fixing the Data Type

In [18]:
# Convert the age into numeric
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
print(df["Age"].dtype)

float64


In [20]:
# Adding median value to NaN
age_median = df["Age"].median()

print("Age Median:", age_median)

df["Age"] = df["Age"].fillna(age_median)

Age Median: 28.0


# Identify the duplicate rows and remove them

In [22]:
# Identify duplicate records while ignoring Employee_ID

duplicate_rows = df.duplicated(
    subset=[
        "Name",
        "Age",
        "Department",
        "Salary",
        "Joining_Date",
        "Experience_Years",
        "Performance_Score"
    ]
)

print("Number of duplicate records:", duplicate_rows.sum())

Number of duplicate records: 1


In [23]:
df = df.drop_duplicates(
    subset=[
        "Name",
        "Age",
        "Department",
        "Salary",
        "Joining_Date",
        "Experience_Years",
        "Performance_Score"
    ]
)

In [24]:
print("Duplicate records after cleaning:", df.duplicated(
    subset=[
        "Name",
        "Age",
        "Department",
        "Salary",
        "Joining_Date",
        "Experience_Years",
        "Performance_Score"
    ]
).sum())

Duplicate records after cleaning: 0


# Identifying invalid values and outliers

In [25]:
print("Age range:")
print("Minimum:", df["Age"].min())
print("Maximum:", df["Age"].max())

Age range:
Minimum: 23.0
Maximum: 150.0


In [33]:
# Clearing the value where age is invalid
import numpy as np
df.loc[df["Age"] > 100, "Age"] = np.nan
print(df[df["Age"] > 100])


Empty DataFrame
Columns: [Employee_ID, Name, Age, Department, Salary, Joining_Date, Experience_Years, Performance_Score]
Index: []


In [34]:
# median of age
age_median = df["Age"].median()

print("Age Median:", age_median)

Age Median: 28.0


In [43]:
# filling the missing value
df["Age"] = df["Age"].fillna(age_median)

In [36]:
print("Age range:")
print("Minimum:", df["Age"].min())
print("Maximum:", df["Age"].max())

Age range:
Minimum: 23.0
Maximum: 41.0


In [27]:
# Invalid values in experience_year
print(df[df["Experience_Years"] < 0])


    Employee_ID   Name   Age Department   Salary Joining_Date  \
17          118  Pooja  28.0         HR  57000.0   2021-11-09   

    Experience_Years  Performance_Score  
17              -2.0                8.0  


In [51]:
# Clearing invalid values in experience_year
df.loc[df["Experience_Years"] < 0, "Experience_Years"] = np.nan

In [52]:
# median of Experience_Years
Experience_Years_median = df["Experience_Years"].median()

print("Experience_Years_Median:", Experience_Years_median)

Experience_Years_Median: 4.5


In [53]:
# Filling Experience_year
df["Experience_Years"] = df["Experience_Years"].fillna(Experience_Years_median)

In [54]:
print(df[df["Age"] > 100])
print(df[df["Experience_Years"] < 0])

Empty DataFrame
Columns: [Employee_ID, Name, Age, Department, Salary, Joining_Date, Experience_Years, Performance_Score]
Index: []
Empty DataFrame
Columns: [Employee_ID, Name, Age, Department, Salary, Joining_Date, Experience_Years, Performance_Score]
Index: []


## Handling the outlier of Salary

In [55]:
# Calculatibg the IQR
Q1 = df["Salary"].quantile(0.25)
Q3 = df["Salary"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

Q1: 49500.0
Q3: 59000.0
IQR: 9500.0
Lower Bound: 35250.0
Upper Bound: 73250.0


In [56]:
salary_outliers = df[
    (df["Salary"] < lower_bound) |
    (df["Salary"] > upper_bound)
]

print("Salary Outliers:")
print(salary_outliers)

Salary Outliers:
    Employee_ID    Name   Age Department    Salary Joining_Date  \
16          117   Nitin  41.0    Finance  500000.0   2017-05-16   
20          121  Manish  37.0    Finance   76000.0   2019-01-21   

    Experience_Years  Performance_Score  
16              18.0                6.9  
20              13.0                7.4  


In [57]:
print("Salary Median:", df["Salary"].median())
print(salary_outliers[["Employee_ID", "Name", "Salary"]])

Salary Median: 53500.0
    Employee_ID    Name    Salary
16          117   Nitin  500000.0
20          121  Manish   76000.0


In [58]:
salary_median = df["Salary"].median()

df.loc[df["Salary"] > 100000, "Salary"] = salary_median

In [59]:
print(df.loc[df["Employee_ID"] == 117, ["Name", "Salary"]])

     Name   Salary
16  Nitin  53500.0


## Replacing the invalid date

In [60]:
# Conveting the invalid date to NaT
df["Joining_Date"] = pd.to_datetime(
    df["Joining_Date"],
    dayfirst=True,
    errors="coerce"
)

/tmp/ipykernel_125083/174827893.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Joining_Date"] = pd.to_datetime(


In [61]:
print(df[df["Joining_Date"].isna()])

    Employee_ID    Name   Age Department   Salary Joining_Date  \
1           102    Riya  27.0         HR  52000.0          NaT   
7           108  Simran  28.0      Sales  55000.0          NaT   
23          124   Nisha  29.0         IT  53000.0          NaT   

    Experience_Years  Performance_Score  
1                4.0                7.5  
7                4.0                7.2  
23               5.0                8.9  


# Confirming the clean data is consistent


In [62]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
Employee_ID          0
Name                 0
Age                  1
Department           0
Salary               0
Joining_Date         3
Experience_Years     0
Performance_Score    0
dtype: int64


In [63]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
Employee_ID          0
Name                 0
Age                  1
Department           0
Salary               0
Joining_Date         3
Experience_Years     0
Performance_Score    0
dtype: int64


In [64]:
duplicate_columns = [
    "Name",
    "Age",
    "Department",
    "Salary",
    "Joining_Date",
    "Experience_Years",
    "Performance_Score"
]

print(
    "Duplicate records:",
    df.duplicated(subset=duplicate_columns).sum()
)

Duplicate records: 0


In [65]:
print("Invalid ages:")
print(df[df["Age"] > 100])

Invalid ages:
Empty DataFrame
Columns: [Employee_ID, Name, Age, Department, Salary, Joining_Date, Experience_Years, Performance_Score]
Index: []


In [66]:
print("Invalid experience values:")
print(df[df["Experience_Years"] < 0])

Invalid experience values:
Empty DataFrame
Columns: [Employee_ID, Name, Age, Department, Salary, Joining_Date, Experience_Years, Performance_Score]
Index: []


In [67]:
print(df[["Employee_ID", "Name", "Salary"]].sort_values("Salary", ascending=False).head())

    Employee_ID    Name   Salary
20          121  Manish  76000.0
8           109  Vikram  72000.0
12          113   Sahil  68000.0
18          119  Aditya  63000.0
4           105   Rahul  61000.0


In [68]:
print("\nFinal Shape:", df.shape)


Final Shape: (25, 8)


In [69]:
print(df)

    Employee_ID    Name   Age Department   Salary Joining_Date  \
0           101    Aman  24.0         IT  45000.0   2023-01-15   
1           102    Riya  27.0         HR  52000.0          NaT   
2           103   Karan  29.0      Sales  48000.0   2021-06-10   
3           104    Neha  25.0         IT  53500.0   2023-03-20   
4           105   Rahul  28.0    Finance  61000.0   2020-11-05   
5           106   Priya  31.0         HR  58000.0   2022-07-18   
6           107   Arjun  26.0         it  47000.0   2023-05-12   
7           108  Simran  28.0      Sales  55000.0          NaT   
8           109  Vikram  35.0    Finance  72000.0   2019-09-30   
9           110  Anjali  23.0         HR  43000.0   2024-01-10   
10          111   Rohit  30.0      Sales  53500.0   2021-04-25   
11          112   Megha  27.0         IT  51000.0   2022-10-14   
12          113   Sahil  32.0    Finance  68000.0   2018-12-01   
13          114   Tanya  26.0         HR  49500.0   2023-06-22   
14        